In [1]:
import numpy as np
import struct
import tensorflow as tf

def load_ubyte_images(filename):
    with open(filename, 'rb') as f:
        magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
        images = np.fromfile(f, dtype=np.uint8).reshape(num, rows, cols)
    return images

def load_ubyte_labels(filename):
    with open(filename, 'rb') as f:
        magic, num = struct.unpack(">II", f.read(8))
        labels = np.fromfile(f, dtype=np.uint8)
    return labels

# Loading your specific files
train_images = load_ubyte_images('MNIST/train-images.idx3-ubyte')
train_labels = load_ubyte_labels('MNIST/train-labels.idx1-ubyte')
test_images = load_ubyte_images('MNIST/t10k-images.idx3-ubyte')
test_labels = load_ubyte_labels('MNIST/t10k-labels.idx1-ubyte')

# Preprocessing: Normalize to [0, 1] and add channel dimension
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0
train_images = np.expand_dims(train_images, -1)
test_images = np.expand_dims(test_images, -1)

print(f"Loaded {len(train_images)} training images.")

Loaded 60000 training images.


In [2]:
# Simple CNN Architecture
model_digits = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model_digits.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_digits.fit(train_images, train_labels, epochs=10, batch_size=64, validation_split=0.1)

# Full Integer Quantization
def representative_data_gen():
    for i in range(100):
        yield [train_images[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model_digits)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# Enforce INT8 logic (Listing 12.6)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_digits = converter.convert()

with open('digit_model.tflite', 'wb') as f:
    f.write(tflite_digits)

Epoch 1/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9147 - loss: 0.3015 - val_accuracy: 0.9723 - val_loss: 0.1008
Epoch 2/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9715 - loss: 0.0976 - val_accuracy: 0.9805 - val_loss: 0.0746
Epoch 3/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9793 - loss: 0.0711 - val_accuracy: 0.9835 - val_loss: 0.0605
Epoch 4/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9835 - loss: 0.0555 - val_accuracy: 0.9840 - val_loss: 0.0567
Epoch 5/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9855 - loss: 0.0472 - val_accuracy: 0.9860 - val_loss: 0.0544
Epoch 6/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9885 - loss: 0.0393 - val_accuracy: 0.9848 - val_loss: 0.0613
Epoch 7/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9893 - loss: 0.0344 - val_accuracy: 0.9873 - val_loss: 0.0519
Epoch 8/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9914 - loss: 0.0285 - val_accuracy: 0.

INFO:tensorflow:Assets written to: C:\Users\ng822\AppData\Local\Temp\tmpjfowsomy\assets


Saved artifact at 'C:\Users\ng822\AppData\Local\Temp\tmpjfowsomy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  1313053858448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1313053859600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1313053859216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1313053859984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1313053859024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1313053860560: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\ng822\anaconda3\envs\421\Lib\site-packages\tensorflow\lite\python\convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [3]:
hex_data = [f'0x{b:02x}' for b in tflite_digits]
with open('digit_model_data.h', 'w') as f:
    f.write('unsigned char digit_model_data[] = {\n')
    f.write(', '.join(hex_data))
    f.write('\n};\n')
    f.write(f'unsigned int digit_model_data_len = {len(tflite_digits)};')

print("digit_model_data.h is ready!")

digit_model_data.h is ready!


In [4]:
interpreter = tf.lite.Interpreter(model_content=tflite_digits)
interpreter.allocate_tensors()
input_idx = interpreter.get_input_details()[0]['index']
output_idx = interpreter.get_output_details()[0]['index']
scale, zp = interpreter.get_input_details()[0]['quantization']

# Test on 100 samples
correct = 0
for i in range(100):
    # Quantize input image
    img = (test_images[i:i+1] / scale + zp).astype(np.int8)
    interpreter.set_tensor(input_idx, img)
    interpreter.invoke()
    if np.argmax(interpreter.get_tensor(output_idx)) == test_labels[i]:
        correct += 1

print(f"Quantized Accuracy: {correct}%")

Quantized Accuracy: 97%


c:\Users\ng822\anaconda3\envs\421\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
